# Prithvi-EO-2.0 + Sen1Floods11 Smoke

This notebook uses Prithvi-EO-2.0-300M non-TL with a 64-sample Sen1Floods11 subset. It runs chip-level classification sanity and a lightweight segmentation fairness smoke. Results are not paper-grade flood segmentation conclusions.


In [ ]:
# 1. Clone or update this repo
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"
REPO_DIR = "rsfm-fairness-audit"
from pathlib import Path
if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}


In [ ]:
# 2. Install package and Prithvi dependencies
!python -m pip install -e .
!python -m pip install -r requirements-prithvi.txt


In [ ]:
# 3. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# 4. Prepare 64 Sen1Floods11 samples directly in Colab local storage
DATA_ROOT = "data/sen1floods11_prithvi_subset64"
CLASS_OUTPUT = "outputs/prithvi_sen1floods11_class64"
SEG_OUTPUT = "outputs/prithvi_sen1floods11_seg64"

!python scripts/prepare_sen1floods11_subset.py \
  --output-dir {DATA_ROOT} \
  --max-samples 64
!head -5 {DATA_ROOT}/metadata.csv


In [ ]:
# 5. Preflight
!python -m rsfm_fairness_audit.cli check-real \
  --dataset sen1floods11 \
  --model prithvi \
  --model-config configs/models/prithvi.yaml \
  --data-root {DATA_ROOT}


In [ ]:
# 6. Chip-level classification sanity
!python -m rsfm_fairness_audit.cli run-real \
  --dataset sen1floods11 \
  --model prithvi \
  --dataset-root {DATA_ROOT} \
  --config configs/models/prithvi.yaml \
  --output-dir {CLASS_OUTPUT} \
  --max-samples 64 \
  --chunk-size 32 \
  --streaming-embeddings true


In [ ]:
# 7. Lightweight segmentation fairness smoke
!python -m rsfm_fairness_audit.cli run-segmentation-real \
  --dataset sen1floods11 \
  --model prithvi \
  --dataset-root {DATA_ROOT} \
  --config configs/models/prithvi.yaml \
  --output-dir {SEG_OUTPUT} \
  --max-samples 64


In [ ]:
# 8. Inspect outputs
!find {CLASS_OUTPUT} -maxdepth 3 -type f -print
!find {SEG_OUTPUT} -maxdepth 3 -type f -print
!sed -n '1,160p' {SEG_OUTPUT}/report.md

from IPython.display import Image, display
fig = f"{SEG_OUTPUT}/figures/segmentation_iou_by_group.png"
if Path(fig).exists():
    display(Image(filename=fig))


In [ ]:
# 9. Package final report artifacts only
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
ZIP_PATH = Path("prithvi_sen1floods11_smoke64_results.zip")
roots = [Path(CLASS_OUTPUT), Path(SEG_OUTPUT)]
files_to_zip = []
for root in roots:
    for rel in ["report.md", "fairness_summary.csv", "raw_vs_balanced_gap.csv", "segmentation_metrics.csv"]:
        path = root / rel
        if path.exists():
            files_to_zip.append(path)
    for folder in [root / "tables", root / "figures"]:
        if folder.exists():
            files_to_zip.extend(sorted(p for p in folder.rglob("*") if p.is_file()))
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for root in roots:
        for path in files_to_zip:
            if path.is_relative_to(root):
                archive.write(path, f"{root.name}/{path.relative_to(root).as_posix()}")
print(f"Packaged {len(files_to_zip)} artifacts into {ZIP_PATH}")
from google.colab import files
files.download(str(ZIP_PATH))
